# Manufacturing Process (PMDCo): from data entry to RDF

This notebook shows how to describe a manufacturing process step and convert
that description into a standardised, machine-readable RDF graph following the
[Platform MaterialDigital Core Ontology (PMDCo)](https://w3id.org/pmd/co/).

**You only need to edit one file:** `docs/example.input.json`. Everything
else is automatic.

A manufacturing process in this schema captures:

- a **name** and optional **description**
- the **input materials** it consumes and the **output materials** it produces
- which **process it follows** in the production chain
- optional **quantitative conditions** such as temperature, pressure, or energy


## What the notebook does

```
example.input.json
  │  your process name, inputs, outputs, and conditions
  │
  ▼
Transform
  │  converts your plain JSON into a structured OO-LD document
  │
  ▼
RDF graph
  │  machine-readable, ontology-aligned triples
  │
  ▼
SHACL validation  →  confirms the graph is structurally correct
SPARQL inspect    →  shows the key connections recorded in the graph
```


## Environment setup

If you are viewing this notebook on GitHub and want to run it locally, clone
the repository first so that all schema files and example inputs are present:

```bash
git clone https://github.com/Semantic-Dataspace/semantic-schemas.git
cd semantic-schemas
```

Then create a virtual environment and start Jupyter:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install semantic-schemas jupyterlab
jupyter lab
```

Open this notebook from the `docs/` subfolder inside JupyterLab.

In [1]:
%pip install -q semantic-schemas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, pathlib, rdflib
from semantic_schemas import Schema

HERE   = pathlib.Path().resolve()   # docs/
SCHEMA = HERE.parent                # process/PMDCo/

schema = Schema(SCHEMA)

print("Schema :", "/".join(SCHEMA.parts[-3:]))


Schema : step/base/PMDCo


## Step 1: Describe your process

Edit `docs/example.input.json` with your data, then run this cell to load it.

| Field | Required | Description |
|---|---|---|
| `process_name` | yes | A name for this process step |
| `description` | no | Free-text explanation of what the process does |
| `inputs` | no | List of IRIs of materials consumed by the process |
| `outputs` | no | List of IRIs of materials produced by the process |
| `preceded_by` | no | List of IRIs of processes that come immediately before this one |
| `conditions` | no | List of quantitative parameters (name, value, unit) |
| `process_id` | no | Custom IRI slug; auto-derived from `process_name` if omitted |

**What is an IRI?**  
An IRI (Internationalized Resource Identifier) is a web address used to uniquely
identify a thing in a knowledge graph. For input and output materials, use the
IRI that was assigned when those materials were registered. For example:
`https://example.org/materials/sintered-magnet-batch-1`.
If materials do not yet have IRIs, leave `inputs` and `outputs` empty for now
and add them later.

In [3]:
simplified = json.loads((HERE / "example.input.json").read_text())

print(json.dumps(simplified, indent=2))

{
  "process_name": "Sintering and Annealing batch 1",
  "description": "Green compacts are heated below the melting point, causing particles to fuse together (sintering). Annealing follows to relieve internal stresses and optimise microstructure.",
  "inputs": [
    "https://example.org/materials/anisotropic-green-compacts-batch-1"
  ],
  "outputs": [
    "https://example.org/materials/sintered-magnet-batch-1"
  ],
  "preceded_by": [
    "https://example.org/processes/pressing-of-fine-powder-batch-1"
  ],
  "conditions": [
    {
      "name": "Sintering Temperature",
      "value": 1080,
      "unit": "\u00b0C"
    },
    {
      "name": "Energy Consumption",
      "value": 26,
      "unit": "kWh"
    }
  ]
}


## Step 2: Convert to OO-LD

This step transforms your plain JSON into a structured
[OO-LD](https://github.com/OO-LD/oold-python) document.
OO-LD is standard JSON enriched with ontology mappings: every field name
is linked to a precise ontology term, so machines can interpret it unambiguously.

The converter also assigns stable identifiers to each node in the graph.
By default, the process identifier is derived from the process name
(e.g. `"Sintering and Annealing batch 1"` → `process-sintering-and-annealing-batch-1`).
You can override this with a custom `process_id` in the input file.

You can also run the transform from the command line:
```
python -m jsonata -e ../specs/transform.simplified.jsonata -i example.input.json
```

In [4]:
oold_doc = schema.transform(simplified)

print(json.dumps(oold_doc, indent=2))

{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/manufacturing/step/base/PMDCo/#v2.0.0",
  "type": "pmdco:PMD_0000029",
  "id": "process-sintering-and-annealing-batch-1",
  "label": "Sintering and Annealing batch 1",
  "description": "Green compacts are heated below the melting point, causing particles to fuse together (sintering). Annealing follows to relieve internal stresses and optimise microstructure.",
  "has_specified_input": [
    "https://example.org/materials/anisotropic-green-compacts-batch-1"
  ],
  "has_specified_output": [
    "https://example.org/materials/sintered-magnet-batch-1"
  ],
  "preceded_by": [
    "https://example.org/processes/pressing-of-fine-powder-batch-1"
  ],
  "has_process_condition": [
    {
      "type": "pmdco:PMD_0000013",
      "id": "process-sintering-and-annealing-batch-1-condition-1",
      "parameter_label": "Sintering Temperature",
      "parameter_value": 1080,
      "parameter_unit": "\u00b0C"
    

## Step 3: Convert to RDF

The OO-LD document is now parsed into an RDF graph.
RDF (Resource Description Framework) is the standard format for knowledge graphs:
everything is expressed as triples of the form *subject, predicate, object*.

The ontology context from `specs/schema.oold.yaml` tells the parser which
ontology IRI each field maps to. For example, `has_specified_input` becomes
`http://purl.obolibrary.org/obo/OBI_0000293`, the canonical OBI term for
the input of a planned process.

In [5]:
flat = schema.to_graph(simplified)

print(f"Graph contains {len(flat)} triples.\n")
print(flat.serialize(format="turtle"))

Graph contains 17 triples.



@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix obo: <http://purl.obolibrary.org/obo/> .
@prefix pmdco: <https://w3id.org/pmd/co/> .
@prefix qudt: <http://qudt.org/schema/qudt/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://w3id.org/pmd/co/test/process-sintering-and-annealing-batch-1> a pmdco:PMD_0000029 ;
    rdfs:label "Sintering and Annealing batch 1" ;
    obo:BFO_0000062 <https://example.org/processes/pressing-of-fine-powder-batch-1> ;
    obo:OBI_0000293 <https://example.org/materials/anisotropic-green-compacts-batch-1> ;
    obo:OBI_0000299 <https://example.org/materials/sintered-magnet-batch-1> ;
    dcterms:conformsTo <https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/manufacturing/step/base/PMDCo/#v2.0.0> ;
    rdfs:comment "Green compacts are heated below the melting point, causing particles to fuse together (sintering). Annealing follows to relieve internal stresses and op

In [6]:
# Optional: save to file
out_ttl = HERE / "output_process.ttl"
out_ttl.write_text(flat.serialize(format="turtle"))
print(f"Written to {out_ttl.name}")

Written to output_process.ttl


## Step 4: Validate against the SHACL shape

SHACL (Shapes Constraint Language) is a way to define rules about what a valid
RDF graph must look like. The shape file `specs/shape.ttl` checks that:

- The process has exactly one label.
- Every input and output is recorded as an IRI (not a plain text string).
- Every process condition has a label, and its value (if given) is a number.

If any rule is violated, the validator prints a clear message explaining what
went wrong.

In [7]:
conforms, violations = schema.validate(flat)

print(f"Conforms: {conforms}")
for v in violations:
    print(f"  Violation: {v}")

Conforms: True


## Step 5: Inspect the graph

The cells below use SPARQL (a query language for RDF graphs) to confirm that
the key facts from your input were recorded correctly.

You do not need to understand SPARQL to use this notebook. Just run the cells
and check that the output matches what you entered.

In [8]:
PMDCO = rdflib.Namespace("https://w3id.org/pmd/co/")
OBI   = rdflib.Namespace("http://purl.obolibrary.org/obo/OBI_")
BFO   = rdflib.Namespace("http://purl.obolibrary.org/obo/BFO_")

# Print process label
proc_iri = next(flat.subjects(rdflib.RDF.type, PMDCO["PMD_0000029"]))
label    = flat.value(proc_iri, rdflib.RDFS.label)
print(f"Process : {proc_iri}")
print(f"Label   : {label}")

Process : https://w3id.org/pmd/co/test/process-sintering-and-annealing-batch-1
Label   : Sintering and Annealing batch 1


In [9]:
SPARQL_IO = """
PREFIX obi:  <http://purl.obolibrary.org/obo/OBI_>
PREFIX bfo:  <http://purl.obolibrary.org/obo/BFO_>
PREFIX pmd:  <https://w3id.org/pmd/co/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?role ?iri
WHERE {
  { ?proc a pmd:PMD_0000029 ; obi:0000293 ?iri . BIND(\"input\"       AS ?role) }
  UNION
  { ?proc a pmd:PMD_0000029 ; obi:0000299 ?iri . BIND(\"output\"      AS ?role) }
  UNION
  { ?proc a pmd:PMD_0000029 ; bfo:0000062 ?iri . BIND(\"preceded by\" AS ?role) }
}
ORDER BY ?role ?iri
"""

rows = list(flat.query(SPARQL_IO))
if rows:
    print(f"{'Role':<14}  IRI")
    print(f"{'':14}  {'':60}")
    for r in rows:
        print(f"{str(r.role):<14}  {r.iri}")
else:
    print("No inputs, outputs, or predecessors recorded.")

Role            IRI
                                                                            
input           https://example.org/materials/anisotropic-green-compacts-batch-1
output          https://example.org/materials/sintered-magnet-batch-1
preceded by     https://example.org/processes/pressing-of-fine-powder-batch-1


In [10]:
SPARQL_COND = """
PREFIX pmd:  <https://w3id.org/pmd/co/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX qudt: <http://qudt.org/schema/qudt/>

SELECT ?name ?value ?unit
WHERE {
  ?proc a pmd:PMD_0000029 ;
        pmd:PMD_0000016 ?cond .
  ?cond rdfs:label ?name .
  OPTIONAL { ?cond qudt:value ?value . }
  OPTIONAL { ?cond qudt:hasUnit ?unit . }
}
ORDER BY ?name
"""

rows = list(flat.query(SPARQL_COND))
if rows:
    print(f"{'Parameter':<30}  {'Value':>10}  Unit")
    print(f"{'':30}  {'':10}  {'':20}")
    for r in rows:
        val  = f"{float(r.value):>10.3f}" if r.value else f"{'n/a':>10}"
        unit = str(r.unit) if r.unit else ""
        print(f"{str(r.name):<30}  {val}  {unit}")
else:
    print("No process conditions recorded.")

Parameter                            Value  Unit
                                                                
Energy Consumption                  26.000  kWh
Sintering Temperature             1080.000  °C


## Summary

| Step | What happens |
|---|---|
| 1 | You fill in `example.input.json` with the process name, materials, and conditions |
| 2 | The data is converted to an OO-LD document (ontology-mapped JSON) |
| 3 | The OO-LD is parsed into an RDF graph |
| 4 | The graph is validated against the SHACL shape |
| 5 | Key facts are extracted from the graph to confirm correctness |

To describe a different process, edit `docs/example.input.json` and re-run all cells.


## Further reading

- [OO-LD primer](../../../docs/2_oold-primer.md): what OO-LD is and how it works
- [Schema format reference](../../../docs/3_schema-format.md): how to write your own schema
- [Simplified input guide](../../../docs/simplified-input-guide.md): end-to-end workflow
- [PMDCo Input and Output of Processes pattern](https://github.com/materialdigital/core-ontology/tree/main/patterns/input%20and%20output%20of%20processes): the ontology pattern this schema follows